<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/JARVIS_PF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Phase field simulation of basic 1D equation,crack propagation and dendritic growth



Allen-Cahn in 1D: Let us consider an example: growth of phase B in phase A. This notebook shows one-dimensional phase-field simulation of the growth of phase B using Allen-Cahn equation.
https://web.tuat.ac.jp/~yamanaka/pcoms2019/Allen-Cahn-1d.html

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import math

# Parameters
nx = 64  # number of computational grids
dx = 0.5e-6  # spacing of computational grid [m]
driving_force = 5.e+5  # magnitude of driving force of growth of phase B: g_A - g_B [J/m3] = [Pa]
interfacial_energy = 1.0  # interfacial energy [J/m2]
interfacial_thickness = 4.0 * dx  # interfacial thickness [m]
interfacial_mobility = 4.e-14  # interfacial mobility [m4/(Js)]
parameter_r = 0.1  # parameter which determines the interfacial area

# Constants
b = 2.0 * np.log((1.0 + (1.0 - 2.0 * parameter_r)) / (1.0 - (1.0 - 2.0 * parameter_r))) / 2.0  # The constant b = 2.1972
a = np.sqrt(3.0 * interfacial_thickness * interfacial_energy / b)  # gradient energy coefficient  "a"[(J/m)^(1/2)]
w = 6.0 * interfacial_energy * b / interfacial_thickness  # potential height W [J/m3]
phase_field_mobility = interfacial_mobility * math.sqrt(2.0 * w) / (6.0 * a)  # mobility of phase-field [m3/(Js)]
dt = dx * dx / (5.0 * phase_field_mobility * a * a) / 2.0  # time increment for a time step [s]



# Finit difference
def do_timestep(nsteps, driv, grad):
    r_nuclei = 10.0 * dx
    p = np.zeros((nsteps, nx))
    for i in range(nx):
        r = i * dx - r_nuclei
        p[0, i] = 0.5 * (1.0 - np.tanh(np.sqrt(2.0 * w) / (2.0 * a) * r))
    p_next = np.copy(p)
    driv_next = np.copy(driv)
    grad_next = np.copy(grad)

    for t in range(nsteps - 1):
        for i in range(nx):
            ip = i + 1
            im = i - 1
            if ip > nx - 1:
                ip = nx - 1
            if im < 0:
                im = 0
            p_next[t + 1, i] = p_next[t, i] + phase_field_mobility * (
                4.0 * w * p_next[t, i] * (1.0 - p_next[t, i]) * (p_next[t, i] - 0.5 + 3.0 / (2.0 * w) * driving_force) +
                a * a * ((p_next[t, ip] - 2 * p_next[t, i] + p_next[t, im]) / (dx * dx))
            ) * dt
            driv_next[t + 1, i] = phase_field_mobility * (
                4.0 * w * p_next[t, i] * (1.0 - p_next[t, i]) * (p_next[t, i] - 0.5 + 3.0 / (2.0 * w) * driving_force)
            )
            grad_next[t + 1, i] = phase_field_mobility * (
                a * a * ((p_next[t, ip] - 2 * p_next[t, i] + p_next[t, im]) / (dx * dx))
            )

    return p_next, driv_next, grad_next




In [ ]:
# Calculate Allen-Cahn equation for "nsteps" times
nsteps=1
p, driv, grad = do_timestep(nsteps,np.zeros((nsteps, nx)), np.zeros((nsteps, nx)))

# Plotting results
fig = plt.figure(figsize=(7, 9))
fig.set_dpi(100)
plt.subplots_adjust(hspace=0.5)

# Phase-field variable plot
ax1 = fig.add_subplot(4, 1, 1)
ax1.set_ylim([0, 1])
ax1.set_xlim([0, nx])
ax1.set_xlabel('Grid number')
ax1.set_ylabel('Phase-field variable')
ax1.grid(True)
ax1.plot(p[-1, :], color="b", marker="o")

# Driving force term plot
ax2 = fig.add_subplot(4, 1, 2)
ax2.set_ylim([-0.03, 0.03])
ax2.set_xlim([0, nx])
ax2.set_xlabel('Grid number')
ax2.set_ylabel('Driving force term')
ax2.grid(True)
ax2.plot(driv[-1, :], color="r", marker="o")

# Gradient term plot
ax3 = fig.add_subplot(4, 1, 3)
ax3.set_ylim([-0.03, 0.03])
ax3.set_xlim([0, nx])
ax3.set_xlabel('Grid number')
ax3.set_ylabel('Gradient term')
ax3.grid(True)
ax3.plot(grad[-1, :], color="y", marker="o")

# Combined driving and gradient term plot
ax4 = fig.add_subplot(4, 1, 4)
ax4.set_xlim([0, nx])
ax4.set_xlabel('Grid number')
ax4.set_ylabel('Driving + Gradient')
ax4.grid(True)
ax4.plot(grad[-1, :] + driv[-1, :], marker="o")

plt.show()

In [ ]:
nsteps=500
p, driv, grad = do_timestep(nsteps, np.zeros((nsteps, nx)), np.zeros((nsteps, nx)))

# Plotting results
fig = plt.figure(figsize=(7, 9))
fig.set_dpi(100)
plt.subplots_adjust(hspace=0.5)

# Phase-field variable plot
ax1 = fig.add_subplot(4, 1, 1)
ax1.set_ylim([0, 1])
ax1.set_xlim([0, nx])
ax1.set_xlabel('Grid number')
ax1.set_ylabel('Phase-field variable')
ax1.grid(True)
ax1.plot(p[-1, :], color="b", marker="o")

# Driving force term plot
ax2 = fig.add_subplot(4, 1, 2)
ax2.set_ylim([-0.03, 0.03])
ax2.set_xlim([0, nx])
ax2.set_xlabel('Grid number')
ax2.set_ylabel('Driving force term')
ax2.grid(True)
ax2.plot(driv[-1, :], color="r", marker="o")

# Gradient term plot
ax3 = fig.add_subplot(4, 1, 3)
ax3.set_ylim([-0.03, 0.03])
ax3.set_xlim([0, nx])
ax3.set_xlabel('Grid number')
ax3.set_ylabel('Gradient term')
ax3.grid(True)
ax3.plot(grad[-1, :], color="y", marker="o")

# Combined driving and gradient term plot
ax4 = fig.add_subplot(4, 1, 4)
ax4.set_xlim([0, nx])
ax4.set_xlabel('Grid number')
ax4.set_ylabel('Driving + Gradient')
ax4.grid(True)
ax4.plot(grad[-1, :] + driv[-1, :], marker="o")

plt.show()

In [ ]:
nsteps=1000
p, driv, grad = do_timestep(nsteps,np.zeros((nsteps, nx)), np.zeros((nsteps, nx)))

# Plotting results
fig = plt.figure(figsize=(7, 9))
fig.set_dpi(100)
plt.subplots_adjust(hspace=0.5)

# Phase-field variable plot
ax1 = fig.add_subplot(4, 1, 1)
ax1.set_ylim([0, 1])
ax1.set_xlim([0, nx])
ax1.set_xlabel('Grid number')
ax1.set_ylabel('Phase-field variable')
ax1.grid(True)
ax1.plot(p[-1, :], color="b", marker="o")

# Driving force term plot
ax2 = fig.add_subplot(4, 1, 2)
ax2.set_ylim([-0.03, 0.03])
ax2.set_xlim([0, nx])
ax2.set_xlabel('Grid number')
ax2.set_ylabel('Driving force term')
ax2.grid(True)
ax2.plot(driv[-1, :], color="r", marker="o")

# Gradient term plot
ax3 = fig.add_subplot(4, 1, 3)
ax3.set_ylim([-0.03, 0.03])
ax3.set_xlim([0, nx])
ax3.set_xlabel('Grid number')
ax3.set_ylabel('Gradient term')
ax3.grid(True)
ax3.plot(grad[-1, :], color="y", marker="o")

# Combined driving and gradient term plot
ax4 = fig.add_subplot(4, 1, 4)
ax4.set_xlim([0, nx])
ax4.set_xlabel('Grid number')
ax4.set_ylabel('Driving + Gradient')
ax4.grid(True)
ax4.plot(grad[-1, :] + driv[-1, :], marker="o")

plt.show()

Phase field simulation of crack propagation 2D: Allen-Cahn vs Cahn-Hillard

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import laplace

# Parameters
nx, ny = 200, 100  # Grid size
dx, dy = 1.0, 1.0  # Grid spacing
dt = 0.1  # Time step size
L = 1.0  # Allen-Cahn parameter
epsilon = 1.0  # Interface width for Allen-Cahn
M = 1  # Mobility for Cahn-Hilliard
kappa = 1  # Gradient energy coefficient for Cahn-Hilliard
num_steps = 1000  # Number of time steps

# Initial condition: a vertical crack in the center for both equations
phi_initial = np.ones((ny, nx))
phi_initial[:, nx//2-5:nx//2+5] = -1

# Initialize order parameter fields
phi_allen_cahn = np.copy(phi_initial)
phi_cahn_hilliard = np.copy(phi_initial)

# Function to compute the Laplacian of phi
def laplacian(phi, dx, dy):
    laplacian_phi = np.zeros_like(phi)
    laplacian_phi[1:-1, 1:-1] = (
        (phi[2:, 1:-1] + phi[:-2, 1:-1] - 2*phi[1:-1, 1:-1]) / dx**2 +
        (phi[1:-1, 2:] + phi[1:-1, :-2] - 2*phi[1:-1, 1:-1]) / dy**2
    )
    return laplacian_phi

# Function to compute the chemical potential for Cahn-Hilliard
def chemical_potential(phi):
    return phi**3 - phi - kappa * laplace(phi)

# Time-stepping loop
for step in range(num_steps):
    # Allen-Cahn update
    laplacian_phi_ac = laplacian(phi_allen_cahn, dx, dy)
    dphi_dt_ac = L * (phi_allen_cahn - phi_allen_cahn**3) - epsilon**2 * laplacian_phi_ac
    phi_allen_cahn += dt * dphi_dt_ac

    # Cahn-Hilliard update
    mu = chemical_potential(phi_cahn_hilliard)
    lap_mu = laplace(mu)
    dphi_dt_ch = M * lap_mu
    phi_cahn_hilliard += dt * dphi_dt_ch

    # Plot results at intervals
    if step % 50 == 0:
        plt.figure(figsize=(12, 5))

        plt.subplot(1, 2, 1)
        plt.imshow(phi_allen_cahn, extent=[0, nx*dx, 0, ny*dy], origin='lower', cmap='RdBu')
        plt.colorbar()
        plt.title(f'Allen-Cahn Equation\nTime step: {step}')
        plt.xlabel('x')
        plt.ylabel('y')

        plt.subplot(1, 2, 2)
        plt.imshow(phi_cahn_hilliard, extent=[0, nx*dx, 0, ny*dy], origin='lower', cmap='RdBu')
        plt.colorbar()
        plt.title(f'Cahn-Hilliard Equation\nTime step: {step}')
        plt.xlabel('x')
        plt.ylabel('y')

        plt.tight_layout()
        plt.pause(0.1)

plt.show()


Install fipy code:

In [ ]:
!pip install -q fipy

In [ ]:
from fipy import Variable, CellVariable, Grid2D, TransientTerm, DiffusionTerm, ImplicitSourceTerm, Viewer, Matplotlib2DGridViewer
from fipy.tools import numerix
dx = dy = 0.025
nx = ny = 100
mesh = Grid2D(dx=dx, dy=dy, nx=nx, ny=ny)

Adapted from: https://www.ctcms.nist.gov/fipy/examples/phase/generated/examples.phase.anisotropy.html

In [ ]:
dt = 5e-4
phase = CellVariable(name=r'$\phi$', mesh=mesh, hasOld=True)
dT = CellVariable(name=r'$\Delta T$', mesh=mesh, hasOld=True)
DT = 2.25
heatEq = (TransientTerm()
          == DiffusionTerm(DT)
          + (phase - phase.old) / dt)
alpha = 0.015
c = 0.02
N = 6.
theta = numerix.pi / 8.
psi = theta + numerix.arctan2(phase.faceGrad[1],
                              phase.faceGrad[0])
Phi = numerix.tan(N * psi / 2)
PhiSq = Phi**2
beta = (1. - PhiSq) / (1. + PhiSq)
DbetaDpsi = -N * 2 * Phi / (1 + PhiSq)
Ddia = (1.+ c * beta)
Doff = c * DbetaDpsi
I0 = Variable(value=((1, 0), (0, 1)))
I1 = Variable(value=((0, -1), (1, 0)))
D = alpha**2 * (1.+ c * beta) * (Ddia * I0 + Doff * I1)
tau = 3e-4
kappa1 = 0.9
kappa2 = 20.
phaseEq = (TransientTerm(tau)
           == DiffusionTerm(D)
           + ImplicitSourceTerm((phase - 0.5 - kappa1 / numerix.pi * numerix.arctan(kappa2 * dT))
                                * (1 - phase)))
radius = dx * 5.
C = (nx * dx / 2, ny * dy / 2)
x, y = mesh.cellCenters
phase.setValue(1., where=((x - C[0])**2 + (y - C[1])**2) < radius**2)


In [ ]:
dT.setValue(-0.5)

In [ ]:
import pylab
class DendriteViewer(Matplotlib2DGridViewer):
    def __init__(self, phase, dT, title=None, limits={}, **kwlimits):
        self.phase = phase
        self.contour = None
        Matplotlib2DGridViewer.__init__(self, vars=(dT,), title=title,
                                        cmap=pylab.cm.hot,
                                        limits=limits, **kwlimits)

    def _plot(self):
        Matplotlib2DGridViewer._plot(self)

        if self.contour is not None:
            for c in self.contour.collections:
                c.remove()

        mesh = self.phase.mesh
        shape = mesh.shape
        x, y = mesh.cellCenters
        z = self.phase.value
        x, y, z = [a.reshape(shape, order='F') for a in (x, y, z)]

        self.contour = self.axes.contour(x, y, z, (0.5,))

viewer = DendriteViewer(phase=phase, dT=dT,
                        title=r"%s & %s" % (phase.name, dT.name),
                        datamin=-0.1, datamax=0.05)

In [ ]:
%%time
steps = 100
from builtins import range
for i in range(steps):
    phase.updateOld()
    dT.updateOld()
    phaseEq.solve(phase, dt=dt)
    heatEq.solve(dT, dt=dt)
    if i==0:
      viewer = DendriteViewer(phase=phaseEq, dT=dT )
    if i%10==0:
      viewer = DendriteViewer(phase=phaseEq, dT=dT )